# Run D: ColBERTv2 Reranking (Google Colab — T4 GPU)

This notebook reranks the top-100 candidates from Run C (BM25F+RM3) using ColBERTv2 late interaction scoring.

**Before running:**
1. Set runtime to GPU: Runtime > Change runtime type > T4 GPU
2. Upload the following two files when prompted in Step 4:
   - `bm25f_rm3.txt` (Run C output, from the `runs/` folder)
   - `docs.jsonl` (full CORD-19 corpus, ~945 MB)

**Output:** `runs/bm25f_rm3_colbert.txt` — download this file after Step 6 completes.

## Step 1: Verify GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type and select T4 GPU."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2: Clone the repository

In [ ]:
import os
import subprocess
from pathlib import Path

# Clone the repo
!git clone https://github.com/hanad28/covid-ir-search-engine.git
os.chdir("covid-ir-search-engine")

print("Working directory:", os.getcwd())
print("Contents:", os.listdir("."))

## Step 3: Install dependencies

In [ ]:
!pip install -r requirements.txt --quiet
!python -m spacy download en_core_web_sm --quiet
!pip install colbert-ai==0.2.21 transformers==4.30.0 --quiet

print("Dependencies installed.")

## Step 4: Upload required files

Upload `bm25f_rm3.txt` and `docs.jsonl` using the cell below. The corpus file (~945 MB) will take a few minutes.

In [ ]:
from google.colab import files
from pathlib import Path

# Create required directories
Path("runs").mkdir(exist_ok=True)
Path("data/corpus_jsonl").mkdir(parents=True, exist_ok=True)

print("Upload bm25f_rm3.txt and docs.jsonl when prompted.")
uploaded = files.upload()

for filename, content in uploaded.items():
    if filename == "bm25f_rm3.txt":
        dest = Path("runs/bm25f_rm3.txt")
    elif filename == "docs.jsonl":
        dest = Path("data/corpus_jsonl/docs.jsonl")
    else:
        print(f"Unexpected file: {filename} — skipping.")
        continue
    dest.write_bytes(content)
    print(f"Saved {filename} to {dest}")

run_c_path = Path("runs/bm25f_rm3.txt")
docs_path = Path("data/corpus_jsonl/docs.jsonl")

print(f"\nRun C exists: {run_c_path.exists()}")
if run_c_path.exists():
    lines = sum(1 for _ in open(run_c_path))
    print(f"  Lines: {lines:,} (expected 30,000)")

print(f"docs.jsonl exists: {docs_path.exists()}")
if docs_path.exists():
    size_gb = docs_path.stat().st_size / 1e9
    print(f"  Size: {size_gb:.2f} GB (expected ~0.95 GB)")

if not (run_c_path.exists() and docs_path.exists()):
    raise FileNotFoundError("Required files missing. Re-run this cell and upload both files.")

## Step 5: Build Lucene index

In [ ]:
import subprocess
from pathlib import Path

INDEX_DIR = Path("index")
INDEX_DIR.mkdir(exist_ok=True)

if not any(INDEX_DIR.iterdir()):
    print("Building Pyserini index for document text lookup...")
    result = subprocess.run(
        [
            "python", "-m", "pyserini.index.lucene",
            "--collection", "JsonCollection",
            "--input", "data/corpus_jsonl",
            "--index", str(INDEX_DIR),
            "--generator", "DefaultLuceneDocumentGenerator",
            "--threads", "4",
            "--storePositions",
            "--storeDocvectors",
            "--storeRaw",
        ],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print("Index build failed:")
        print(result.stderr)
    else:
        print("Index built successfully.")
else:
    print("Index already exists, skipping.")

## Step 6: Run ColBERTv2 reranking

In [ ]:
import sys
import json
import torch
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# Hardcoded config values
INDEX_DIR = Path("index")
COLBERT_MODEL = "colbert-ir/colbertv2.0"
RERANK_DEPTH = 100
RUN_NAMES = {
    "C": "bm25f_rm3",
    "D": "bm25f_rm3_colbert",
}
RUNS_DIR = Path("runs")

from colbert.modeling.checkpoint import Checkpoint
from colbert.infra import ColBERTConfig
from pyserini.search.lucene import LuceneSearcher


def load_document_text(doc_id: str, lucene_searcher) -> str:
    """Load title + abstract for a document from the Lucene index."""
    try:
        raw = lucene_searcher.doc(doc_id).raw()
        doc = json.loads(raw)
        title = doc.get("title", "").strip()
        abstract = doc.get("abstract", "").strip()
        return f"{title} {abstract}".strip()
    except Exception:
        return ""


def get_rerank_candidates(run_path: Path, depth: int = RERANK_DEPTH) -> dict:
    """Extract top-N document IDs per topic from a TREC runfile."""
    candidates = {}
    with open(run_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 6:
                continue
            topic_id, _, doc_id, rank, _, _ = parts
            if int(rank) > depth:
                continue
            if topic_id not in candidates:
                candidates[topic_id] = []
            candidates[topic_id].append(doc_id)
    return candidates


def save_reranked_results(results: dict, output_path: Path, run_name: str) -> None:
    """Write reranked results to TREC runfile format."""
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w") as f:
        for topic_id, doc_scores in sorted(results.items(), key=lambda x: int(x[0])):
            for rank, (doc_id, score) in enumerate(doc_scores, start=1):
                f.write(f"{topic_id} Q0 {doc_id} {rank} {score:.6f} {run_name}\n")

    total = sum(len(v) for v in results.values())
    print(f"Saved {total:,} reranked results to {output_path}")


def get_queries(preprocess_text: bool = True) -> dict:
    """Return {topic_number: query_string} for all topics."""
    import xml.etree.ElementTree as ET
    
    topics_file = Path("data/topics/topics-rnd1.xml")
    tree = ET.parse(topics_file)
    root = tree.getroot()
    
    queries = {}
    for topic_el in root.findall("topic"):
        topic_num = topic_el.attrib["number"]
        query = topic_el.findtext("query", default="").strip()
        question = topic_el.findtext("question", default="").strip()
        raw = f"{query} {question}".strip()
        queries[topic_num] = raw
    
    return queries


def rerank_with_colbert(queries: dict, candidates: dict, model_name: str = COLBERT_MODEL) -> dict:
    """Rerank candidates using ColBERTv2 MaxSim scoring."""
    checkpoint = Checkpoint(model_name, colbert_config=ColBERTConfig())
    lucene_searcher = LuceneSearcher(str(INDEX_DIR))

    results = {}
    for topic_id, query_text in tqdm(queries.items(), desc="Reranking topics"):
        if topic_id not in candidates:
            continue

        doc_ids = candidates[topic_id]
        doc_texts = [load_document_text(did, lucene_searcher) for did in doc_ids]

        # Encode query and documents independently (late interaction)
        query_embs = checkpoint.queryFromText([query_text], bsize=1, to_cpu=True)
        doc_embs = checkpoint.docFromText(doc_texts, bsize=32, to_cpu=True)

        query_emb = query_embs[0]  # shape: (query_len, dim)
        doc_scores = []
        for i, doc_emb in enumerate(doc_embs):
            # MaxSim: for each query token, take max similarity across all doc tokens
            sim_matrix = torch.matmul(query_emb, doc_emb.T)
            maxsim_score = sim_matrix.max(dim=1).values.sum().item()
            doc_scores.append((doc_ids[i], maxsim_score))

        doc_scores.sort(key=lambda x: x[1], reverse=True)
        results[topic_id] = doc_scores

    lucene_searcher.close()
    return results


# Load queries (raw text for neural model, no preprocessing)
queries = get_queries(preprocess_text=False)
print(f"Loaded {len(queries)} queries")

# Load Run C candidates
run_c_path = RUNS_DIR / f"{RUN_NAMES['C']}.txt"
candidates = get_rerank_candidates(run_c_path, depth=RERANK_DEPTH)
print(f"Loaded top-{RERANK_DEPTH} candidates for {len(candidates)} topics")

# Rerank
print(f"Reranking with {COLBERT_MODEL}...")
reranked = rerank_with_colbert(queries, candidates)
print(f"Reranked {len(reranked)} topics")

# Save
output_path = RUNS_DIR / f"{RUN_NAMES['D']}.txt"
save_reranked_results(reranked, output_path, RUN_NAMES["D"])
print(f"Run D saved to {output_path}")

## Step 7: Download the output

Run the cell below to download `runs/bm25f_rm3_colbert.txt` to your local machine. Place it in the `runs/` folder of the project, then run `scripts/run_rrf.py` and `scripts/evaluate.py` to complete the pipeline.

In [ ]:
from google.colab import files
from pathlib import Path

output_path = Path("runs/bm25f_rm3_colbert.txt")

if output_path.exists():
    line_count = sum(1 for _ in open(output_path))
    print(f"Run D file generated: {line_count:,} lines (expected 3,000 for 30 topics x 100 docs)")
    files.download(str(output_path))
else:
    print("Error: runs/bm25f_rm3_colbert.txt not found. Check Step 6 for errors.")